In [1]:
!pip install scikit-learn pandas numpy joblib streamlit

In [3]:
from google.colab import files
uploaded = files.upload()

Saving AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv to AuthentiText_X_2026_AI_vs_Human_Detection_1K (3).csv


In [12]:
import pandas as pd

df = pd.read_csv('AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (1000, 12)
Columns: ['text_id', 'content_text', 'author_type', 'model_source', 'prompt_complexity_score', 'perplexity_score', 'burstiness_index', 'syntactic_variability', 'semantic_coherence_score', 'lexical_diversity_ratio', 'readability_grade_level', 'generation_confidence_score']


,text_id,content_text,author_type,model_source,prompt_complexity_score,perplexity_score,burstiness_index,syntactic_variability,semantic_coherence_score,lexical_diversity_ratio,readability_grade_level,generation_confidence_score
0,TXT_0001,learning pattern detection algorithm pattern n...,AI,Human,0.029,73.75,0.953,0.465,0.351,0.187,12.2,0.162
1,TXT_0002,algorithm algorithm data research network mode...,Human,Claude,0.605,43.11,0.054,0.952,0.314,0.636,9.8,0.012
2,TXT_0003,analysis language generation research pattern ...,Human,GPT-4,0.396,59.97,0.709,0.945,0.684,0.500,13.5,0.171
3,TXT_0004,data language system learning content data net...,AI,GPT-4,0.299,18.99,0.532,0.780,0.216,0.103,12.9,0.838
4,TXT_0005,model learning content language model generati...,AI,Human,0.867,82.45,0.478,0.602,0.420,0.198,6.4,0.022


In [13]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['content_text'].apply(clean_text)
df[['content_text', 'clean_text']].head()

,content_text,clean_text
0,learning pattern detection algorithm pattern n...,learning pattern detection algorithm pattern n...
1,algorithm algorithm data research network mode...,algorithm algorithm data research network mode...
2,analysis language generation research pattern ...,analysis language generation research pattern ...
3,data language system learning content data net...,data language system learning content data net...
4,model learning content language model generati...,model learning content language model generati...


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['author_type'])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df['label'].value_counts())

Label mapping: {'AI': np.int64(0), 'Human': np.int64(1)}
label
1    537
0    463
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 800
Test size: 200


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Feature matrix shape:", X_train_tfidf.shape)

Feature matrix shape: (800, 182)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Training complete!")

Training complete!


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

preds = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.465

Classification Report:
               precision    recall  f1-score   support

           0       0.43      0.34      0.38        97
           1       0.48      0.58      0.53       103

    accuracy                           0.47       200
   macro avg       0.46      0.46      0.46       200
weighted avg       0.46      0.47      0.46       200


Confusion Matrix:
 [[33 64]
 [43 60]]


In [10]:
print(df['label'].value_counts())
print("Predicted:", pd.Series(preds).value_counts())
print("Actual:", pd.Series(y_test).value_counts())

label
1    537
0    463
Name: count, dtype: int64
Predicted: 1    124
0     76
Name: count, dtype: int64
Actual: label
1    103
0     97
Name: count, dtype: int64


In [11]:
def predict_text(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    label_name = le.inverse_transform([pred])[0]
    return f"Prediction: {label_name}"

sample = "One thing that excites me about AI is how it can simplify everyday work. I enjoy exploring tools like ChatGPT and using them to automate repetitive tasks, organize information."
print(predict_text(sample))

Prediction: Human
